# Step 14 — resolve multipart child blocks and recalculate population

**# of cells in notebook:** 4

**Purpose:** Identify selected child-block features that contain multiple disconnected polygon parts, convert them into appropriate singlepart child blocks, and recalculate population where geometry has changed. Completely surrounded multipart pieces are absorbed into the surrounding child block rather than being retained as separate block features. Population totals are then compared with the original selected outputs as a final QA check.

**Input:**

- `heterogeneous_largePop_selection`, containing the selected and ID-assigned:
  - `new_blocks_populated.gpkg`
- a population geodatabase containing:
  - `pop_grid`
  - `buildings_inside`

**Output:**

- `heterogeneous_largePop_blocks_MPexplode`
- within affected block folders:
  - revised `new_blocks_populated.gpkg`
- multipart-processing summary CSV
- population-assignment diagnostic CSVs
- population-update summary CSV
- original-versus-MPexplode population comparison CSVs

**Main logic:**

**Cell 1 — Inventory multipart selected blocks**

1. Scans the selected block GeoPackages.
2. Distinguishes true multipart geometries from single polygons, polygons with holes, and one-part MultiPolygons.
3. Identifies the source-block folders and layers containing true multipart child features.
4. Reports the affected blocks for subsequent processing.

**Cell 2 — Explode multipart features and absorb surrounded pieces**

1. Processes the affected selected block layers.
2. Examines each component of every multipart child block.
3. If a component is completely covered by another child feature or lies entirely within another feature's interior hole, absorbs it into that surrounding feature rather than creating a disconnected child block.
4. Explodes the remaining disconnected components into individual child-block features.
5. Adds suffixes to `block_id` values where multiple child parts are created.
6. Clears geometry-dependent fields that are no longer valid after geometry changes, including `cell_area_m2`, `area_m_utm`, and `population`; `cluster_smooth` is also cleared for newly exploded children.
7. Writes the revised block GeoPackages and a multipart-processing summary.

**Cell 3 — Recalculate population after multipart processing**

1. Reads the revised MPexplode block features.
2. Selects relevant population-grid cells and buildings.
3. Calculates building-area shares for each revised child block within each population-grid cell.
4. Reapportions grid-cell population among the revised child geometries.
5. Updates the `population` field while preserving the other existing fields.
6. Writes per-layer population QA CSVs and an overall population-update summary.

This cell recalculates population for child features created or geometrically changed by the multipart workflow.

**Cell 4 — Compare population before and after multipart processing**

1. Pairs each MPexplode output with its corresponding original selected layer.
2. Compares child-level and total population values.
3. Writes summary and detail CSVs.
4. Identifies blocks whose total population changed beyond the specified numerical tolerance.
5. Raises an error by default if any before/after total-population discrepancy exceeds the tolerance, after first writing the diagnostic CSVs.


In [ ]:
from pathlib import Path

import numpy as np
import pyogrio
import shapely


# ---------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------

ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection"
)


# ---------------------------------------------------------------------
# Functions
# ---------------------------------------------------------------------

def natural_block_key(path: Path):
    """Sort names such as _136, _390, and _2740 numerically."""
    try:
        return 0, int(path.name.lstrip("_"))
    except ValueError:
        return 1, path.name.lower()


def count_multipart_features(
    gpkg_path: Path,
    layer_name: str,
) -> int:
    """
    Count features containing more than one actual geometry component.

    Examples:
      Polygon                          -> 1 component, not multipart
      Polygon with interior holes      -> 1 component, not multipart
      MultiPolygon with one polygon    -> 1 component, not multipart
      MultiPolygon with two polygons   -> 2 components, multipart
    """

    # columns=[] means read geometry but no attribute fields.
    gdf = pyogrio.read_dataframe(
        gpkg_path,
        layer=layer_name,
        columns=[],
        read_geometry=True,
    )

    if gdf.empty:
        return 0

    # Simple geometries return 1. Multipart geometries return their
    # number of component geometries. Null geometries return 0.
    part_counts = shapely.get_num_geometries(gdf.geometry.array)

    return int(np.count_nonzero(part_counts > 1))


# ---------------------------------------------------------------------
# Main diagnostic
# ---------------------------------------------------------------------

if not ROOT.is_dir():
    raise FileNotFoundError(f"Root directory does not exist:\n{ROOT}")

multipart_blocks = []
errors = []

block_folders = sorted(
    (path for path in ROOT.iterdir() if path.is_dir()),
    key=natural_block_key,
)

print(f"Scanning {len(block_folders)} block folder(s)...\n")

for block_folder in block_folders:

    block_has_multipart = False

    # Search recursively in case a GeoPackage is below another subfolder.
    gpkg_files = sorted(
        path
        for path in block_folder.rglob("*")
        if path.is_file() and path.suffix.lower() == ".gpkg"
    )

    if not gpkg_files:
        print(f"[NO GPKG]   {block_folder.name}")
        continue

    for gpkg_path in gpkg_files:

        try:
            layers = pyogrio.list_layers(gpkg_path)

        except Exception as exc:
            errors.append(
                (
                    block_folder.name,
                    str(gpkg_path),
                    "<could not list layers>",
                    str(exc),
                )
            )
            print(
                f"[ERROR]     {block_folder.name} | "
                f"{gpkg_path.name} | could not list layers"
            )
            continue

        for layer_name, geometry_type in layers:

            # pyogrio.list_layers() also returns nonspatial tables.
            if geometry_type is None:
                continue

            layer_name = str(layer_name)

            try:
                multipart_count = count_multipart_features(
                    gpkg_path,
                    layer_name,
                )

            except Exception as exc:
                errors.append(
                    (
                        block_folder.name,
                        str(gpkg_path),
                        layer_name,
                        str(exc),
                    )
                )
                print(
                    f"[ERROR]     {block_folder.name} | "
                    f"{gpkg_path.name} | {layer_name}"
                )
                continue

            if multipart_count > 0:
                block_has_multipart = True

                print(
                    f"[MULTIPART] {block_folder.name} | "
                    f"{gpkg_path.name} | "
                    f"{layer_name} | "
                    f"{multipart_count} feature(s)"
                )

    if block_has_multipart:
        multipart_blocks.append(block_folder.name)


# ---------------------------------------------------------------------
# Final list
# ---------------------------------------------------------------------

print("\n" + "=" * 78)
print("BLOCK FOLDERS CONTAINING MULTIPART FEATURES")
print("=" * 78)

if multipart_blocks:
    for folder_name in multipart_blocks:
        print(folder_name)
else:
    print("None found.")

print("\nPython list:")
print(multipart_blocks)


# ---------------------------------------------------------------------
# Errors
# ---------------------------------------------------------------------

if errors:
    print("\n" + "=" * 78)
    print(f"WARNING: {len(errors)} item(s) could not be checked")
    print("=" * 78)

    for block_name, gpkg_name, layer_name, message in errors:
        print(
            f"{block_name} | {gpkg_name} | "
            f"{layer_name} | {message}"
        )


In [ ]:
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import geopandas as gpd
import pyogrio
import shapely

from shapely.geometry import Polygon
from shapely.geometry.base import BaseGeometry
from shapely.ops import unary_union


# ---------------------------------------------------------------------
# Inputs
# ---------------------------------------------------------------------

IN_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection"
)

OUT_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks_MPexplode"
)

SUMMARY_CSV = OUT_ROOT / "multipart_explode_absorb_summary.csv"

OVERWRITE_OUTPUT_GPKGS = True


# ---------------------------------------------------------------------
# Behavior settings
# ---------------------------------------------------------------------

ID_FIELD = "block_id"

# Fields to blank for newly exploded rows.
CLEAR_FIELDS_FOR_EXPLODED_PARTS = [
    "cluster_smooth",
    "cell_area_m2",
    "area_m_utm",
    "population",
]

# Fields to blank when an existing feature's geometry changes because
# it absorbed a surrounded island or lost one of its own parts.
#
# I am not blanking cluster_smooth here by default, because the recipient
# feature is not a newly created exploded child feature. Its geometry changed,
# though, so area and population fields are no longer reliable.
CLEAR_FIELDS_FOR_GEOMETRY_CHANGED_SINGLEPARTS = [
    "cell_area_m2",
    "area_m_utm",
    "population",
]

# Multipart resolution changes geometry, not the reason/method by which the
# feature entered the split workflow.

# If a multipart feature loses surrounded island parts and only one part remains,
# this controls whether the remaining feature keeps its original block_id or gets
# a suffix like blk_401_2_0_2_1.
#
# False is usually cleaner:
#   blk_401_2_0_2 stays blk_401_2_0_2 if only one piece remains.
#
# True is more literal:
#   blk_401_2_0_2 becomes blk_401_2_0_2_1.
SUFFIX_SINGLE_REMAINING_PART_AFTER_ABSORB = False

# Absorption tests.
#
# The first two should usually be True.
# The third one is intentionally False because it is more permissive and can
# absorb parts that are inside a feature's outer shell but not necessarily inside
# a true interior hole.
ABSORB_IF_COVERED_BY_OTHER_FEATURE = True
ABSORB_IF_INSIDE_HOLE_OF_OTHER_FEATURE = True
ABSORB_IF_INSIDE_OUTER_SHELL_ONLY = False

# Optional safety valve.
# Example: set to 0.25 if you only want to absorb parts that are less than 25%
# of the original multipart feature's total area.
# Leave as None to absorb any completely surrounded part.
MAX_ABSORBED_PART_SHARE_OF_SOURCE = None

# Set True only if invalid geometries are causing overlay/union errors.
FIX_INVALID_GEOMETRIES = False


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def natural_block_key(path: Path):
    """Sort folders like _393, _401, _10615 numerically."""
    try:
        return 0, int(path.name.lstrip("_"))
    except ValueError:
        return 1, path.name.lower()


def make_valid_if_requested(geom: BaseGeometry):
    if geom is None or geom.is_empty:
        return geom

    if not FIX_INVALID_GEOMETRIES:
        return geom

    try:
        if not geom.is_valid:
            return shapely.make_valid(geom)
    except Exception:
        return geom

    return geom


def get_parts(geom: BaseGeometry):
    """
    Return actual component geometries.

    Polygon with holes = one part.
    MultiPolygon with two polygons = two parts.
    """
    if geom is None or geom.is_empty:
        return []

    geom = make_valid_if_requested(geom)

    n = shapely.get_num_geometries(geom)

    if n <= 1:
        return [geom]

    return list(shapely.get_parts(geom))


def polygon_components(geom: BaseGeometry):
    """
    Return Polygon components from Polygon, MultiPolygon, or GeometryCollection.
    """
    if geom is None or geom.is_empty:
        return []

    geom = make_valid_if_requested(geom)

    geom_type = geom.geom_type

    if geom_type == "Polygon":
        return [geom]

    if geom_type == "MultiPolygon":
        return list(geom.geoms)

    if geom_type == "GeometryCollection":
        polys = []
        for part in geom.geoms:
            polys.extend(polygon_components(part))
        return polys

    return []


def union_geometries(geometries):
    """
    Robust-ish union helper for a list of geometries.
    """
    clean = [
        make_valid_if_requested(g)
        for g in geometries
        if g is not None and not g.is_empty
    ]

    if not clean:
        return None

    if len(clean) == 1:
        return clean[0]

    try:
        return shapely.union_all(clean)
    except Exception:
        return unary_union(clean)


def prepare_nullable_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """
    Convert integer columns that may receive nulls to pandas nullable Int64.
    """
    gdf = gdf.copy()

    fields_that_may_receive_nulls = sorted(
        set(CLEAR_FIELDS_FOR_EXPLODED_PARTS)
        .union(CLEAR_FIELDS_FOR_GEOMETRY_CHANGED_SINGLEPARTS)
    )

    for col in fields_that_may_receive_nulls:
        if col in gdf.columns and pd.api.types.is_integer_dtype(gdf[col]):
            gdf[col] = gdf[col].astype("Int64")

    return gdf


def force_original_like_dtypes(
    out_gdf: gpd.GeoDataFrame,
    original_gdf: gpd.GeoDataFrame,
) -> gpd.GeoDataFrame:
    """
    Try to preserve nullable integer fields after building the output.
    """
    out_gdf = out_gdf.copy()

    for col in original_gdf.columns:
        if col == original_gdf.geometry.name:
            continue

        if col not in out_gdf.columns:
            continue

        if pd.api.types.is_integer_dtype(original_gdf[col]):
            out_gdf[col] = pd.to_numeric(
                out_gdf[col],
                errors="coerce",
            ).astype("Int64")

    return out_gdf


def clear_fields(row: pd.Series, fields_to_clear):
    """
    Set selected fields to null if they exist.
    """
    row = row.copy()

    for col in fields_to_clear:
        if col in row.index:
            row[col] = pd.NA

    return row


def layer_has_multipart(gdf: gpd.GeoDataFrame) -> bool:
    """
    True if any feature contains more than one actual geometry part.
    """
    if gdf.empty:
        return False

    part_counts = shapely.get_num_geometries(gdf.geometry.array)
    return bool(np.any(part_counts > 1))


def candidate_surrounds_part(
    part_geom: BaseGeometry,
    candidate_geom: BaseGeometry,
):
    """
    Return possible surround matches for candidate_geom.

    Returns a list of dictionaries. Empty list means candidate does not
    surround the part.

    We check:
      1. candidate geometry covers the part
      2. part is inside one of candidate's interior holes
      3. optional: part is inside candidate's outer shell only
    """
    matches = []

    if (
        part_geom is None
        or part_geom.is_empty
        or candidate_geom is None
        or candidate_geom.is_empty
    ):
        return matches

    part_geom = make_valid_if_requested(part_geom)
    candidate_geom = make_valid_if_requested(candidate_geom)

    # Case 1: true geometry containment/coverage.
    # This catches cases where the candidate polygon actually covers the part.
    if ABSORB_IF_COVERED_BY_OTHER_FEATURE:
        try:
            if candidate_geom.covers(part_geom):
                matches.append({
                    "method": "covered_by_candidate_geometry",
                    "priority": 1,
                    "score_area": float(candidate_geom.area),
                })
        except Exception:
            pass

    # Case 2: the part is inside a hole of the candidate polygon.
    # This catches the common GIS case shown in your screenshot, where the
    # orange feature has an interior ring around the green island.
    if ABSORB_IF_INSIDE_HOLE_OF_OTHER_FEATURE:
        for poly in polygon_components(candidate_geom):
            for hole_i, interior_ring in enumerate(poly.interiors, start=1):
                try:
                    hole_poly = Polygon(interior_ring)

                    if hole_poly.is_empty:
                        continue

                    if FIX_INVALID_GEOMETRIES and not hole_poly.is_valid:
                        hole_poly = shapely.make_valid(hole_poly)

                    if hole_poly.covers(part_geom):
                        matches.append({
                            "method": f"inside_candidate_interior_hole_{hole_i}",
                            "priority": 0,
                            "score_area": float(hole_poly.area),
                        })

                except Exception:
                    continue

    # Case 3: optional permissive shell test.
    # This can be useful for messy data but can also be too aggressive.
    if ABSORB_IF_INSIDE_OUTER_SHELL_ONLY:
        for poly_i, poly in enumerate(polygon_components(candidate_geom), start=1):
            try:
                shell_poly = Polygon(poly.exterior)

                if shell_poly.is_empty:
                    continue

                if FIX_INVALID_GEOMETRIES and not shell_poly.is_valid:
                    shell_poly = shapely.make_valid(shell_poly)

                if shell_poly.covers(part_geom):
                    matches.append({
                        "method": f"inside_candidate_outer_shell_{poly_i}",
                        "priority": 2,
                        "score_area": float(shell_poly.area),
                    })

            except Exception:
                continue

    return matches


def find_absorbing_feature(
    part_geom: BaseGeometry,
    source_idx,
    source_geom: BaseGeometry,
    gdf: gpd.GeoDataFrame,
    spatial_index,
):
    """
    Find another feature that completely surrounds this part.

    If several features qualify, choose the most specific one:
      - interior-hole containment is preferred
      - then direct coverage
      - then optional outer-shell containment
      - smaller containing area wins
    """
    if part_geom is None or part_geom.is_empty:
        return None

    if MAX_ABSORBED_PART_SHARE_OF_SOURCE is not None:
        try:
            source_area = float(source_geom.area)
            if source_area > 0:
                share = float(part_geom.area) / source_area
                if share > MAX_ABSORBED_PART_SHARE_OF_SOURCE:
                    return None
        except Exception:
            pass

    try:
        candidate_positions = list(spatial_index.query(part_geom))
    except Exception:
        candidate_positions = range(len(gdf))

    choices = []

    for pos in candidate_positions:
        candidate_idx = gdf.index[pos]

        if candidate_idx == source_idx:
            continue

        candidate_geom = gdf.geometry.iloc[pos]

        matches = candidate_surrounds_part(
            part_geom=part_geom,
            candidate_geom=candidate_geom,
        )

        if not matches:
            continue

        candidate_row = gdf.iloc[pos]
        candidate_block_id = (
            candidate_row.get(ID_FIELD, None)
            if ID_FIELD in candidate_row.index
            else None
        )

        for m in matches:
            choices.append({
                "absorber_idx": candidate_idx,
                "absorber_block_id": candidate_block_id,
                "method": m["method"],
                "priority": m["priority"],
                "score_area": m["score_area"],
                "candidate_area": float(candidate_geom.area),
            })

    if not choices:
        return None

    choices = sorted(
        choices,
        key=lambda x: (
            x["priority"],
            x["score_area"],
            x["candidate_area"],
            str(x["absorber_idx"]),
        ),
    )

    return choices[0]


def make_output_row(
    source_row: pd.Series,
    geom: BaseGeometry,
    block_id,
    fields_to_clear,
    geom_col,
):
    new_row = source_row.copy()
    new_row[geom_col] = geom

    if ID_FIELD in new_row.index:
        if pd.isna(block_id):
            new_row[ID_FIELD] = pd.NA
        else:
            new_row[ID_FIELD] = block_id

    new_row = clear_fields(new_row, fields_to_clear)

    return new_row


# ---------------------------------------------------------------------
# Explode with absorption
# ---------------------------------------------------------------------

def explode_multipart_layer(
    gdf: gpd.GeoDataFrame,
    block_folder_name: str,
    gpkg_name: str,
    layer_name: str,
):
    """
    Explode multipart features, but absorb surrounded multipart parts into
    the feature that surrounds them.

    Returns:
      out_gdf, summary_rows, stats
    """

    gdf = prepare_nullable_columns(gdf)

    geom_col = gdf.geometry.name
    spatial_index = gdf.sindex

    original_parts_by_idx = {}
    absorbed_part_orders_by_source = defaultdict(set)
    additions_by_absorber = defaultdict(list)

    summary_rows = []

    n_multipart_features = 0
    n_absorbed_parts = 0

    # ------------------------------------------------------------
    # First pass:
    # Identify multipart parts that should be absorbed into another
    # feature rather than written as their own exploded child.
    # ------------------------------------------------------------

    for source_idx, row in gdf.iterrows():
        source_geom = row[geom_col]
        source_parts = get_parts(source_geom)
        original_parts_by_idx[source_idx] = source_parts

        if len(source_parts) <= 1:
            continue

        n_multipart_features += 1

        source_block_id = (
            row.get(ID_FIELD, None)
            if ID_FIELD in row.index
            else None
        )

        for part_order, part_geom in enumerate(source_parts, start=1):
            absorber = find_absorbing_feature(
                part_geom=part_geom,
                source_idx=source_idx,
                source_geom=source_geom,
                gdf=gdf,
                spatial_index=spatial_index,
            )

            if absorber is None:
                continue

            absorber_idx = absorber["absorber_idx"]

            absorbed_part_orders_by_source[source_idx].add(part_order)
            additions_by_absorber[absorber_idx].append(part_geom)
            n_absorbed_parts += 1

            summary_rows.append({
                "action": "absorbed_into_surrounding_feature",
                "block_folder": block_folder_name,
                "gpkg": gpkg_name,
                "layer": layer_name,
                "source_fid": source_idx,
                "source_block_id": source_block_id,
                "source_part_order": part_order,
                "absorber_fid": absorber_idx,
                "absorber_block_id": absorber["absorber_block_id"],
                "new_block_id": absorber["absorber_block_id"],
                "part_rank_area_desc": None,
                "original_part_order": part_order,
                "part_area": float(part_geom.area),
                "absorption_method": absorber["method"],
                "note": (
                    "Part was not written as a separate row; "
                    "it was unioned into the surrounding feature."
                ),
            })

    # ------------------------------------------------------------
    # Second pass:
    # Build output rows.
    # ------------------------------------------------------------

    output_rows = []
    n_output_exploded_rows = 0
    n_rows_dropped_all_parts_absorbed = 0
    n_geometry_changed_singlepart_rows = 0

    for source_idx, row in gdf.iterrows():
        source_geom = row[geom_col]
        original_parts = original_parts_by_idx.get(
            source_idx,
            get_parts(source_geom),
        )

        source_block_id = (
            row.get(ID_FIELD, None)
            if ID_FIELD in row.index
            else None
        )

        absorbed_orders = absorbed_part_orders_by_source.get(source_idx, set())
        additions = additions_by_absorber.get(source_idx, [])

        kept_own_parts = [
            part
            for part_order, part in enumerate(original_parts, start=1)
            if part_order not in absorbed_orders
        ]

        final_geom = union_geometries(kept_own_parts + additions)

        row_lost_parts = len(absorbed_orders) > 0
        row_gained_parts = len(additions) > 0
        row_was_multipart = len(original_parts) > 1
        row_geometry_changed = row_lost_parts or row_gained_parts

        if final_geom is None or final_geom.is_empty:
            n_rows_dropped_all_parts_absorbed += 1

            summary_rows.append({
                "action": "dropped_all_parts_absorbed",
                "block_folder": block_folder_name,
                "gpkg": gpkg_name,
                "layer": layer_name,
                "source_fid": source_idx,
                "source_block_id": source_block_id,
                "source_part_order": None,
                "absorber_fid": None,
                "absorber_block_id": None,
                "new_block_id": None,
                "part_rank_area_desc": None,
                "original_part_order": None,
                "part_area": None,
                "absorption_method": None,
                "note": (
                    "All parts from this source feature were absorbed into "
                    "surrounding features, so no row was written for it."
                ),
            })

            continue

        final_parts = get_parts(final_geom)

        # If only one final part remains, write one row.
        # Usually this keeps the original block_id.
        if len(final_parts) <= 1:
            final_part = final_parts[0]

            out_block_id = source_block_id

            if (
                SUFFIX_SINGLE_REMAINING_PART_AFTER_ABSORB
                and row_was_multipart
                and source_block_id is not None
                and not pd.isna(source_block_id)
            ):
                out_block_id = f"{source_block_id}_1"

            fields_to_clear = []

            if row_geometry_changed or row_was_multipart:
                fields_to_clear = CLEAR_FIELDS_FOR_GEOMETRY_CHANGED_SINGLEPARTS
                n_geometry_changed_singlepart_rows += 1

            new_row = make_output_row(
                source_row=row,
                geom=final_part,
                block_id=out_block_id,
                fields_to_clear=fields_to_clear,
                geom_col=geom_col,
            )

            output_rows.append(new_row)

            if row_was_multipart or row_geometry_changed:
                summary_rows.append({
                    "action": "written_as_singlepart_after_absorb_or_union",
                    "block_folder": block_folder_name,
                    "gpkg": gpkg_name,
                    "layer": layer_name,
                    "source_fid": source_idx,
                    "source_block_id": source_block_id,
                    "source_part_order": None,
                    "absorber_fid": None,
                    "absorber_block_id": None,
                    "new_block_id": out_block_id,
                    "part_rank_area_desc": 1,
                    "original_part_order": None,
                    "part_area": float(final_part.area),
                    "absorption_method": None,
                    "note": (
                        "Feature ended as one singlepart geometry. "
                        "Original block_id was kept unless suffix setting "
                        "was enabled."
                    ),
                })

            continue

        # If more than one final part remains, explode to singlepart rows.
        # Largest part gets suffix _1.
        parts_sorted = sorted(
            enumerate(final_parts),
            key=lambda item: (-item[1].area, item[0]),
        )

        for part_rank, (final_part_order, part_geom) in enumerate(
            parts_sorted,
            start=1,
        ):
            if pd.isna(source_block_id):
                out_block_id = pd.NA
            else:
                out_block_id = f"{source_block_id}_{part_rank}"

            new_row = make_output_row(
                source_row=row,
                geom=part_geom,
                block_id=out_block_id,
                fields_to_clear=CLEAR_FIELDS_FOR_EXPLODED_PARTS,
                geom_col=geom_col,
            )

            output_rows.append(new_row)
            n_output_exploded_rows += 1

            summary_rows.append({
                "action": "written_as_exploded_singlepart",
                "block_folder": block_folder_name,
                "gpkg": gpkg_name,
                "layer": layer_name,
                "source_fid": source_idx,
                "source_block_id": source_block_id,
                "source_part_order": None,
                "absorber_fid": None,
                "absorber_block_id": None,
                "new_block_id": out_block_id,
                "part_rank_area_desc": part_rank,
                "original_part_order": final_part_order + 1,
                "part_area": float(part_geom.area),
                "absorption_method": None,
                "note": (
                    "Final geometry still had multiple parts, so it was "
                    "exploded. Largest final part receives suffix _1."
                ),
            })

    if output_rows:
        out_gdf = gpd.GeoDataFrame(
            output_rows,
            columns=gdf.columns,
            geometry=geom_col,
            crs=gdf.crs,
        )
    else:
        out_gdf = gpd.GeoDataFrame(
            columns=gdf.columns,
            geometry=geom_col,
            crs=gdf.crs,
        )

    out_gdf = force_original_like_dtypes(out_gdf, gdf)

    # Do not write the old dataframe index into the output GeoPackage.
    out_gdf = out_gdf.reset_index(drop=True)

    stats = {
        "input_features": len(gdf),
        "output_features": len(out_gdf),
        "net_row_change": len(out_gdf) - len(gdf),
        "multipart_features": n_multipart_features,
        "absorbed_parts": n_absorbed_parts,
        "output_exploded_rows": n_output_exploded_rows,
        "geometry_changed_singlepart_rows": n_geometry_changed_singlepart_rows,
        "rows_dropped_all_parts_absorbed": n_rows_dropped_all_parts_absorbed,
    }

    return out_gdf, summary_rows, stats


# ---------------------------------------------------------------------
# Main workflow
# ---------------------------------------------------------------------

if not IN_ROOT.is_dir():
    raise FileNotFoundError(f"Input root does not exist:\n{IN_ROOT}")

OUT_ROOT.mkdir(parents=True, exist_ok=True)

all_summary_rows = []
affected_block_folders = []

block_folders = sorted(
    [p for p in IN_ROOT.iterdir() if p.is_dir()],
    key=natural_block_key,
)

print(f"Scanning {len(block_folders)} block folder(s)...\n")

for block_folder in block_folders:

    gpkg_files = sorted(
        p for p in block_folder.rglob("*.gpkg") if p.is_file()
    )

    if not gpkg_files:
        continue

    checked_layers = []
    block_has_multipart = False

    for gpkg_path in gpkg_files:

        try:
            layers = pyogrio.list_layers(gpkg_path)
        except Exception as exc:
            print(f"[ERROR] Could not list layers: {gpkg_path}")
            print(f"        {exc}")
            continue

        for layer_name, geometry_type in layers:

            # Skip nonspatial tables.
            if geometry_type is None:
                continue

            layer_name = str(layer_name)

            try:
                gdf = pyogrio.read_dataframe(
                    gpkg_path,
                    layer=layer_name,
                    fid_as_index=True,
                )
            except Exception as exc:
                print(
                    f"[ERROR] Could not read layer: "
                    f"{block_folder.name} | {gpkg_path.name} | {layer_name}"
                )
                print(f"        {exc}")
                continue

            has_mp = layer_has_multipart(gdf)

            checked_layers.append({
                "gpkg_path": gpkg_path,
                "layer_name": layer_name,
                "gdf": gdf,
                "has_multipart": has_mp,
            })

            if has_mp:
                block_has_multipart = True

    # If this block folder has no multipart features, do not create output.
    if not block_has_multipart:
        continue

    affected_block_folders.append(block_folder.name)

    out_block_folder = OUT_ROOT / block_folder.name
    out_block_folder.mkdir(parents=True, exist_ok=True)

    print(f"[BLOCK] {block_folder.name}")

    # Group checked layers by GeoPackage so we can recreate each affected GPKG.
    gpkg_groups = {}

    for item in checked_layers:
        gpkg_groups.setdefault(item["gpkg_path"], []).append(item)

    for gpkg_path, layer_items in gpkg_groups.items():

        out_gpkg = out_block_folder / gpkg_path.name

        if out_gpkg.exists():
            if OVERWRITE_OUTPUT_GPKGS:
                out_gpkg.unlink()
            else:
                raise FileExistsError(
                    f"Output GeoPackage already exists:\n{out_gpkg}"
                )

        wrote_any_layer = False

        for item in layer_items:

            layer_name = item["layer_name"]
            gdf = item["gdf"]

            if item["has_multipart"]:
                out_gdf, summary_rows, stats = explode_multipart_layer(
                    gdf=gdf,
                    block_folder_name=block_folder.name,
                    gpkg_name=gpkg_path.name,
                    layer_name=layer_name,
                )

                all_summary_rows.extend(summary_rows)

                print(
                    f"  [EXPLODE/ABSORB] {gpkg_path.name} | {layer_name} | "
                    f"{stats['multipart_features']} multipart feature(s), "
                    f"{stats['absorbed_parts']} absorbed part(s), "
                    f"rows {stats['input_features']} -> {stats['output_features']} "
                    f"({stats['net_row_change']:+})"
                )

            else:
                # Copy unchanged spatial layers from this GPKG.
                out_gdf = gdf.reset_index(drop=True)

                print(
                    f"  [COPIED]         {gpkg_path.name} | {layer_name} | "
                    f"no multipart features"
                )

            pyogrio.write_dataframe(
                out_gdf,
                out_gpkg,
                layer=layer_name,
                driver="GPKG",
            )

            wrote_any_layer = True

        if wrote_any_layer:
            print(f"  [WROTE]          {out_gpkg}")

    print()


# ---------------------------------------------------------------------
# Write summary
# ---------------------------------------------------------------------

summary_df = pd.DataFrame(all_summary_rows)

if not summary_df.empty:
    summary_df.to_csv(SUMMARY_CSV, index=False)
    print(f"Summary CSV written:\n{SUMMARY_CSV}")
else:
    print("No multipart features were found.")

print("\n" + "=" * 78)
print("AFFECTED BLOCK FOLDERS")
print("=" * 78)

for folder_name in affected_block_folders:
    print(folder_name)

print("\nPython list:")
print(affected_block_folders)


In [ ]:
# -*- coding: utf-8 -*-
r"""
Update population field in exploded multipart block outputs.

This version is adapted for:

    E:\_johannesburg\_analysis\heterogeneous_largePop_blocks_MPexplode

Each block folder is expected to contain:

    new_blocks_populated.gpkg

The script:
    - reads the existing layer(s) in each GeoPackage
    - recalculates population using pop_grid + buildings_inside
    - updates ONLY the population column
    - preserves all other existing columns
    - backs up each GeoPackage before replacing it
    - writes per-layer diagnostic CSVs
    - writes one overall summary CSV

No ArcPy required.
"""

import re
import csv
import shutil
import traceback
from pathlib import Path
from collections import defaultdict
from datetime import datetime

import pandas as pd
import geopandas as gpd
import pyogrio


# ============================================================
# USER SETTINGS
# ============================================================

large_pop_blocks_folder = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks_MPexplode"
)

population_gdb = Path(
    r"E:\_johannesburg\_analysis\population_wp2\population_wp2.gdb"
)

target_gpkg_name = "new_blocks_populated.gpkg"

pop_grid_layer = "pop_grid"
buildings_layer = "buildings_inside"

population_field = "population"
area_field = "area_m_utm_new"
pop_field = "grid_code"

create_backups = True
overwrite_existing_backups = False

fix_invalid_geometries = False
area_tolerance = 1e-9

log_file = large_pop_blocks_folder / "MPexplode_population_update.log"
summary_csv = large_pop_blocks_folder / "MPexplode_population_update_summary.csv"

verbose_console = False


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def msg(text="", also_console=None):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    line = f"[{timestamp}] {text}"

    try:
        with open(log_file, "a", encoding="utf-8") as log:
            log.write(line + "\n")
    except Exception:
        pass

    if also_console is None:
        also_console = verbose_console

    if also_console:
        print(text, flush=True)


def safe_float(value):
    if value is None:
        return 0.0
    try:
        if pd.isna(value):
            return 0.0
    except Exception:
        pass
    try:
        return float(value)
    except Exception:
        return 0.0


def sanitize_name(name):
    name = re.sub(r"[^A-Za-z0-9_]", "_", str(name))
    if re.match(r"^[0-9]", name):
        name = "x_" + name
    return name


def get_k_suffix(layer_name):
    parts = str(layer_name).split("_")
    last = parts[-1]
    if last.isdigit():
        return f"k{last}"
    return "kall"


def list_layer_info(dataset_path):
    layers = pyogrio.list_layers(str(dataset_path))
    out = []

    for row in layers:
        layer_name = str(row[0])
        geometry_type = row[1] if len(row) > 1 else None
        out.append((layer_name, geometry_type))

    return out


def list_layer_names(dataset_path):
    return [name for name, _geom_type in list_layer_info(dataset_path)]


def layer_exists(dataset_path, layer_name):
    lower = layer_name.lower()
    return lower in {lyr.lower() for lyr in list_layer_names(dataset_path)}


def actual_layer_name(dataset_path, layer_name):
    lower = layer_name.lower()
    for lyr in list_layer_names(dataset_path):
        if lyr.lower() == lower:
            return lyr
    raise RuntimeError(f"Layer not found: {layer_name} in {dataset_path}")


def require_columns(df, required_columns, label):
    existing_lower = {c.lower() for c in df.columns}
    missing = [c for c in required_columns if c.lower() not in existing_lower]

    if missing:
        raise RuntimeError(f"{label} is missing required column(s): {missing}")


def read_vector(path, layer, columns=None, bbox=None, read_geometry=True):
    kwargs = {
        "layer": layer,
        "columns": columns,
        "bbox": bbox,
        "read_geometry": read_geometry,
    }

    kwargs = {k: v for k, v in kwargs.items() if v is not None}

    try:
        return pyogrio.read_dataframe(str(path), fid_as_index=True, **kwargs)
    except TypeError:
        return pyogrio.read_dataframe(str(path), **kwargs)


def add_stable_id_from_index(gdf, id_field):
    out = gdf.copy()
    out[id_field] = out.index.to_series(index=out.index).astype(str).values
    out = out.reset_index(drop=True)
    return out


def geometry_union(gdf):
    try:
        return gdf.geometry.union_all()
    except Exception:
        return gdf.geometry.unary_union


def make_valid_if_requested(gdf, label):
    if not fix_invalid_geometries:
        return gdf

    msg(f"    Repairing invalid geometries: {label}")
    out = gdf.copy()
    out["geometry"] = out.geometry.make_valid()
    return out


def is_projected_crs(gdf):
    try:
        return bool(gdf.crs and gdf.crs.is_projected)
    except Exception:
        return False


def ensure_same_crs(gdf, target_crs, label):
    if gdf.crs is None:
        raise RuntimeError(f"{label} has unknown CRS.")

    if target_crs is None:
        raise RuntimeError("Target block layer has unknown CRS.")

    if gdf.crs != target_crs:
        msg(f"    Reprojecting {label} to match block layer CRS.")
        return gdf.to_crs(target_crs)

    return gdf


def iter_block_folders(base_folder):
    for item in sorted(base_folder.iterdir()):
        if item.is_dir() and item.name.startswith("_"):
            yield item


def is_block_layer_name(layer_name):
    """
    Handles both normal pyogrio names like:
        blk_1_401_5

    and ArcGIS display names that may appear like:
        main.blk_1_401_5
    """
    name = str(layer_name).lower()
    return name.startswith("blk_") or ".blk_" in name


def get_spatial_layers(gpkg_path):
    return [
        name
        for name, geometry_type in list_layer_info(gpkg_path)
        if geometry_type is not None
    ]


def get_layers_to_update(gpkg_path):
    spatial_layers = get_spatial_layers(gpkg_path)

    block_layers = [
        layer_name
        for layer_name in spatial_layers
        if is_block_layer_name(layer_name)
    ]

    # Your GeoPackages should have one spatial layer.
    # This fallback allows the script to run even if the layer name
    # does not start with blk_ for some reason.
    if not block_layers and len(spatial_layers) == 1:
        block_layers = spatial_layers

    return sorted(block_layers)


def read_block_layers_for_folder(gpkg_path, layer_names):
    block_layers = {}

    for layer_name in layer_names:
        gdf = read_vector(gpkg_path, layer=layer_name)

        if gdf.empty:
            msg(f"  Warning: {layer_name} is empty.")

        gdf = gdf.reset_index(drop=True).copy()

        # Temporary ID only for this run. It is dropped before writing.
        gdf["_tmp_pop_block_fid"] = range(1, len(gdf) + 1)

        block_layers[layer_name] = gdf

    return block_layers


def combined_bounds(block_layers):
    bounds = []

    for gdf in block_layers.values():
        if not gdf.empty:
            bounds.append(gdf.total_bounds)

    if not bounds:
        return None

    b = pd.DataFrame(bounds, columns=["minx", "miny", "maxx", "maxy"])

    return (
        float(b["minx"].min()),
        float(b["miny"].min()),
        float(b["maxx"].max()),
        float(b["maxy"].max()),
    )


def read_population_candidates(folder_block_layers):
    first_gdf = next(iter(folder_block_layers.values()))
    target_crs = first_gdf.crs

    if target_crs is None:
        raise RuntimeError("Block layer has unknown CRS.")

    bbox = combined_bounds(folder_block_layers)

    if bbox is None:
        raise RuntimeError("No non-empty block layers found for folder.")

    msg("  Reading candidate pop_grid cells from FileGDB...")
    msg(f"    bbox: {bbox}")

    pop_layer_actual = actual_layer_name(population_gdb, pop_grid_layer)

    pop_candidates = read_vector(
        population_gdb,
        layer=pop_layer_actual,
        columns=[pop_field],
        bbox=bbox,
    )

    if pop_candidates.empty:
        return pop_candidates, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    pop_candidates = ensure_same_crs(
        pop_candidates,
        target_crs,
        "pop_grid",
    )

    require_columns(pop_candidates, [pop_field], "pop_grid")

    pop_candidates = add_stable_id_from_index(
        pop_candidates,
        "_grid_fid",
    )

    all_blocks = pd.concat(
        [
            gdf[["geometry"]]
            for gdf in folder_block_layers.values()
            if not gdf.empty
        ],
        ignore_index=True,
    )

    all_blocks = gpd.GeoDataFrame(
        all_blocks,
        geometry="geometry",
        crs=target_crs,
    )

    all_blocks_union = geometry_union(all_blocks)

    pop_selected = pop_candidates[
        pop_candidates.geometry.intersects(all_blocks_union)
    ].copy()

    msg(f"  Candidate pop_grid cells read: {len(pop_candidates):,}")
    msg(f"  Selected pop_grid cells:       {len(pop_selected):,}")

    if pop_selected.empty:
        return pop_selected, gpd.GeoDataFrame(geometry=[], crs=target_crs)

    grid_bbox = tuple(float(v) for v in pop_selected.total_bounds)

    msg("  Reading candidate buildings_inside features from FileGDB...")
    msg(f"    selected grid bbox: {grid_bbox}")

    bldg_layer_actual = actual_layer_name(population_gdb, buildings_layer)

    try:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            columns=[],
            bbox=grid_bbox,
        )
    except Exception:
        buildings = read_vector(
            population_gdb,
            layer=bldg_layer_actual,
            bbox=grid_bbox,
        )

    buildings = ensure_same_crs(
        buildings,
        target_crs,
        "buildings_inside",
    )

    grid_union = geometry_union(pop_selected)

    buildings = buildings[
        buildings.geometry.intersects(grid_union)
    ].copy()

    buildings = buildings[["geometry"]].copy()

    msg(f"  Buildings selected: {len(buildings):,}")

    return pop_selected, buildings


def write_assigned_population_csv(
    csv_path,
    block_fid_field,
    block_ids,
    block_assigned_population,
):
    if csv_path.exists():
        csv_path.unlink()

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow([block_fid_field, "assigned_population"])

        for block_id in sorted(block_ids):
            assigned_pop = safe_float(
                block_assigned_population.get(int(block_id), 0.0)
            )
            writer.writerow([block_id, assigned_pop])


def write_grid_detail_csv(csv_path, block_fid_field, grid_detail_rows):
    if csv_path.exists():
        csv_path.unlink()

    with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)

        writer.writerow([
            "FID_pop_grid_selection",
            "grid_code",
            block_fid_field,
            "group_built_area_m2",
            "total_built_area_m2_in_grid",
            "built_area_share",
            "apportioned_population",
            "note",
        ])

        for r in grid_detail_rows:
            writer.writerow([
                r["FID_pop_grid_selection"],
                r["grid_code"],
                r["block_fid"],
                r["group_built_area_m2"],
                r["total_built_area_m2_in_grid"],
                r["built_area_share"],
                r["apportioned_population"],
                r["note"],
            ])


# ============================================================
# CORE POPULATION ASSIGNMENT
# ============================================================

def calculate_population_for_block_layer(
    blocks_gdf,
    block_layer_name,
    pop_selected,
    buildings,
    block_folder,
):
    safe_block_name = sanitize_name(block_layer_name)
    k_suffix = get_k_suffix(block_layer_name)

    msg("")
    msg("------------------------------------------------------------")
    msg(f"Processing block layer: {block_layer_name}", also_console=True)
    msg(f"  k suffix: {k_suffix}")
    msg("------------------------------------------------------------")

    assigned_pop_csv = (
        block_folder / f"{safe_block_name}_MPexplode_assigned_population.csv"
    )

    grid_detail_csv = (
        block_folder / f"{safe_block_name}_MPexplode_grid_apportionment_detail.csv"
    )

    blocks = blocks_gdf.copy()

    if blocks.empty:
        blocks[population_field] = 0.0

        write_assigned_population_csv(
            assigned_pop_csv,
            "block_fid",
            [],
            {},
        )

        write_grid_detail_csv(
            grid_detail_csv,
            "block_fid",
            [],
        )

        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": 0,
            "building_intersections": 0,
            "grid_building_block_intersections": 0,
            "block_features_updated": 0,
            "zero_population_features": 0,
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": "empty block layer",
        }

    if not is_projected_crs(blocks):
        msg("  WARNING: block layer CRS does not appear to be projected.")
        msg("  Area calculations may not be in square meters.")

    blocks = make_valid_if_requested(blocks, "blocks")
    pop_selected = make_valid_if_requested(pop_selected, "pop_grid")
    buildings = make_valid_if_requested(buildings, "buildings")

    block_union = geometry_union(blocks[["geometry"]])

    pop_for_layer = pop_selected[
        pop_selected.geometry.intersects(block_union)
    ].copy()

    selected_count = len(pop_for_layer)

    msg(f"Selected pop_grid cells for this layer: {selected_count:,}")

    def finish_zero(status):
        blocks[population_field] = 0.0

        write_assigned_population_csv(
            assigned_pop_csv,
            block_fid_field="block_fid",
            block_ids=list(blocks["_tmp_pop_block_fid"]),
            block_assigned_population={},
        )

        write_grid_detail_csv(
            grid_detail_csv,
            block_fid_field="block_fid",
            grid_detail_rows=[],
        )

        return blocks, {
            "block_layer": block_layer_name,
            "k": k_suffix,
            "selected_grid_cells": selected_count,
            "building_intersections": 0,
            "grid_building_block_intersections": 0,
            "block_features_updated": len(blocks),
            "zero_population_features": len(blocks),
            "total_grid_population_seen": 0.0,
            "total_population_assigned_to_blocks": 0.0,
            "status": status,
        }

    if selected_count == 0:
        msg("WARNING: No pop_grid cells selected. Population will be set to 0.")
        return finish_zero("no pop_grid cells selected")

    layer_grid_union = geometry_union(pop_for_layer)

    buildings_for_layer = buildings[
        buildings.geometry.intersects(layer_grid_union)
    ].copy()

    if buildings_for_layer.empty:
        msg("WARNING: No buildings intersect selected pop_grid cells. Population will be set to 0.")
        return finish_zero("no buildings in selected pop_grid cells")

    msg("Running overlay: selected pop_grid cells ∩ buildings...")

    grid_bldg = gpd.overlay(
        pop_for_layer[["_grid_fid", pop_field, "geometry"]],
        buildings_for_layer[["geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if grid_bldg.empty:
        msg("WARNING: Grid-building overlay produced no features. Population will be set to 0.")
        return finish_zero("no grid-building intersections")

    grid_bldg[area_field] = grid_bldg.geometry.area
    grid_bldg = grid_bldg[
        grid_bldg[area_field] > area_tolerance
    ].copy()

    building_intersections_count = len(grid_bldg)

    msg(f"Grid-building intersection features: {building_intersections_count:,}")

    if grid_bldg.empty:
        return finish_zero("all grid-building intersections had zero area")

    grid_total_built_area = (
        grid_bldg.groupby("_grid_fid", dropna=False)[area_field]
        .sum()
        .to_dict()
    )

    grid_population = (
        grid_bldg.groupby("_grid_fid", dropna=False)[pop_field]
        .first()
        .apply(safe_float)
        .to_dict()
    )

    msg("Running overlay: grid-building pieces ∩ block polygons...")

    block_bldg = gpd.overlay(
        grid_bldg[["_grid_fid", "geometry"]],
        blocks[["_tmp_pop_block_fid", "geometry"]],
        how="intersection",
        keep_geom_type=False,
    )

    if block_bldg.empty:
        block_bldg[area_field] = []
    else:
        block_bldg[area_field] = block_bldg.geometry.area
        block_bldg = block_bldg[
            block_bldg[area_field] > area_tolerance
        ].copy()

    msg(f"Grid-building-block intersection features: {len(block_bldg):,}")

    if block_bldg.empty:
        grid_block_built_area = pd.DataFrame(
            columns=["_grid_fid", "_tmp_pop_block_fid", area_field]
        )
    else:
        grid_block_built_area = (
            block_bldg
            .groupby(["_grid_fid", "_tmp_pop_block_fid"], dropna=False)[area_field]
            .sum()
            .reset_index()
        )

    msg("Apportioning grid-cell population to block features...")

    block_assigned_population = defaultdict(float)
    grid_detail_rows = []

    inside_area_by_grid = defaultdict(dict)
    inside_total_by_grid = defaultdict(float)

    for _, r in grid_block_built_area.iterrows():
        grid_id = r["_grid_fid"]
        block_fid = int(r["_tmp_pop_block_fid"])
        group_area = safe_float(r[area_field])

        inside_area_by_grid[grid_id][block_fid] = group_area
        inside_total_by_grid[grid_id] += group_area

    zero_area_grid_count = 0

    for grid_id in sorted(grid_total_built_area.keys(), key=lambda x: str(x)):
        total_area = safe_float(grid_total_built_area[grid_id])
        grid_pop = safe_float(grid_population.get(grid_id, 0.0))

        if total_area <= 0:
            zero_area_grid_count += 1

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": None,
                "group_built_area_m2": 0.0,
                "total_built_area_m2_in_grid": 0.0,
                "built_area_share": 0.0,
                "apportioned_population": 0.0,
                "note": "zero total built area in grid",
            })

            continue

        for block_fid, group_area in sorted(
            inside_area_by_grid.get(grid_id, {}).items()
        ):
            built_area_share = group_area / total_area
            apportioned_pop = grid_pop * built_area_share

            block_assigned_population[block_fid] += apportioned_pop

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": block_fid,
                "group_built_area_m2": group_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": built_area_share,
                "apportioned_population": apportioned_pop,
                "note": "inside block",
            })

        outside_area = total_area - safe_float(
            inside_total_by_grid.get(grid_id, 0.0)
        )

        if outside_area > area_tolerance:
            outside_share = outside_area / total_area
            outside_pop = grid_pop * outside_share

            grid_detail_rows.append({
                "FID_pop_grid_selection": grid_id,
                "grid_code": grid_pop,
                "block_fid": -1,
                "group_built_area_m2": outside_area,
                "total_built_area_m2_in_grid": total_area,
                "built_area_share": outside_share,
                "apportioned_population": outside_pop,
                "note": "outside blocks",
            })

    msg(f"Grid cells with zero total built area: {zero_area_grid_count:,}")
    msg(f"Block features receiving population: {len(block_assigned_population):,}")

    write_assigned_population_csv(
        assigned_pop_csv,
        block_fid_field="block_fid",
        block_ids=list(blocks["_tmp_pop_block_fid"]),
        block_assigned_population=block_assigned_population,
    )

    write_grid_detail_csv(
        grid_detail_csv,
        block_fid_field="block_fid",
        grid_detail_rows=grid_detail_rows,
    )

    blocks[population_field] = blocks["_tmp_pop_block_fid"].map(
        lambda x: float(block_assigned_population.get(int(x), 0.0))
    )

    updated_count = len(blocks)
    zero_count = int((blocks[population_field] == 0).sum())

    total_grid_population_seen = sum(
        safe_float(v)
        for v in grid_population.values()
    )

    total_population_assigned_to_blocks = sum(
        safe_float(v)
        for v in block_assigned_population.values()
    )

    msg("Layer done.")
    msg("Summary:")
    msg(f"  Block layer:                         {block_layer_name}")
    msg(f"  Selected grid cells:                 {selected_count:,}")
    msg(f"  Grid-building intersections:         {building_intersections_count:,}")
    msg(f"  Grid-building-block intersections:   {len(block_bldg):,}")
    msg(f"  Block features updated:              {updated_count:,}")
    msg(f"  Block features assigned zero pop:    {zero_count:,}")
    msg(f"  Total grid population represented:   {total_grid_population_seen}")
    msg(f"  Total population assigned to blocks: {total_population_assigned_to_blocks}")

    return blocks, {
        "block_layer": block_layer_name,
        "k": k_suffix,
        "selected_grid_cells": selected_count,
        "building_intersections": building_intersections_count,
        "grid_building_block_intersections": len(block_bldg),
        "block_features_updated": updated_count,
        "zero_population_features": zero_count,
        "total_grid_population_seen": total_grid_population_seen,
        "total_population_assigned_to_blocks": total_population_assigned_to_blocks,
        "status": "success",
    }


# ============================================================
# GPKG WRITING
# ============================================================

def make_backup(gpkg_path):
    if not create_backups:
        return None

    backup_path = gpkg_path.with_name(
        gpkg_path.stem + "_backup_before_population_update.gpkg"
    )

    if backup_path.exists():
        if overwrite_existing_backups:
            backup_path.unlink()
        else:
            msg(f"  Backup already exists, leaving it as-is:")
            msg(f"    {backup_path}")
            return backup_path

    shutil.copy2(gpkg_path, backup_path)

    msg(f"  Backup created:")
    msg(f"    {backup_path}")

    return backup_path


def write_updated_gpkg(gpkg_path, layer_gdfs):
    """
    Rewrites the GeoPackage through a temporary file, then replaces the original.

    This is safer than trying to overwrite a layer inside the same GeoPackage
    while it is open.
    """
    tmp_gpkg = gpkg_path.with_name(gpkg_path.stem + "_tmp_population_update.gpkg")

    if tmp_gpkg.exists():
        tmp_gpkg.unlink()

    for layer_name, gdf in layer_gdfs.items():
        out_gdf = gdf.copy()

        if "_tmp_pop_block_fid" in out_gdf.columns:
            out_gdf = out_gdf.drop(columns=["_tmp_pop_block_fid"])

        out_gdf = out_gdf.reset_index(drop=True)

        out_gdf.to_file(
            str(tmp_gpkg),
            layer=layer_name,
            driver="GPKG",
            engine="pyogrio",
        )

    gpkg_path.unlink()
    tmp_gpkg.replace(gpkg_path)


# ============================================================
# MAIN
# ============================================================

def main():
    if log_file.exists():
        log_file.unlink()

    msg("Starting MPexplode population update", also_console=True)
    msg(f"Block folder root: {large_pop_blocks_folder}", also_console=True)
    msg(f"Population GDB:    {population_gdb}", also_console=True)
    msg(f"Target GPKG name:  {target_gpkg_name}", also_console=True)
    msg(f"Log file:          {log_file}", also_console=True)
    msg("", also_console=True)

    if not large_pop_blocks_folder.is_dir():
        raise FileNotFoundError(
            "large_pop_blocks_folder does not exist:\n"
            f"{large_pop_blocks_folder}"
        )

    if not population_gdb.exists():
        raise FileNotFoundError(
            "Population geodatabase does not exist:\n"
            f"{population_gdb}"
        )

    if not layer_exists(population_gdb, pop_grid_layer):
        raise FileNotFoundError(
            f"Population grid layer '{pop_grid_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    if not layer_exists(population_gdb, buildings_layer):
        raise FileNotFoundError(
            f"Buildings layer '{buildings_layer}' does not exist in:\n"
            f"{population_gdb}"
        )

    block_folders = list(iter_block_folders(large_pop_blocks_folder))

    msg("Block folders found:", also_console=True)
    msg(f"  {len(block_folders)}", also_console=True)

    overall_summary = []

    for block_folder in block_folders:
        block_folder_name = block_folder.name
        gpkg_path = block_folder / target_gpkg_name

        msg("")
        msg("============================================================")
        msg(f"Block folder: {block_folder_name}", also_console=True)
        msg("============================================================")

        if not gpkg_path.exists():
            msg("Skipping folder because target GeoPackage does not exist:")
            msg(f"  {gpkg_path}")
            continue

        try:
            layers_to_update = get_layers_to_update(gpkg_path)

            if not layers_to_update:
                msg("No spatial block layers found in:")
                msg(f"  {gpkg_path}")

                overall_summary.append({
                    "block_folder": block_folder_name,
                    "gpkg": gpkg_path.name,
                    "block_layer": None,
                    "k": None,
                    "selected_grid_cells": None,
                    "building_intersections": None,
                    "grid_building_block_intersections": None,
                    "block_features_updated": None,
                    "zero_population_features": None,
                    "total_grid_population_seen": None,
                    "total_population_assigned_to_blocks": None,
                    "status": "no spatial block layers found",
                })

                continue

            msg("Layers to update:")
            for layer_name in layers_to_update:
                msg(f"  {layer_name}", also_console=True)

            make_backup(gpkg_path)

            block_layers = read_block_layers_for_folder(
                gpkg_path,
                layers_to_update,
            )

            pop_selected, buildings = read_population_candidates(block_layers)

            updated_layers = {}

            # Read all spatial layers so the rewritten GeoPackage keeps them.
            # In your current structure, this should usually be one layer.
            all_spatial_layers = get_spatial_layers(gpkg_path)

            for layer_name in all_spatial_layers:
                if layer_name not in layers_to_update:
                    unchanged_gdf = read_vector(gpkg_path, layer=layer_name)
                    updated_layers[layer_name] = unchanged_gdf.reset_index(drop=True)
                    continue

                try:
                    updated_gdf, result = calculate_population_for_block_layer(
                        blocks_gdf=block_layers[layer_name],
                        block_layer_name=layer_name,
                        pop_selected=pop_selected,
                        buildings=buildings,
                        block_folder=block_folder,
                    )

                    updated_layers[layer_name] = updated_gdf

                    result["block_folder"] = block_folder_name
                    result["gpkg"] = gpkg_path.name
                    overall_summary.append(result)

                except Exception as e:
                    msg("")
                    msg("ERROR while processing layer:")
                    msg(f"  Folder: {block_folder_name}")
                    msg(f"  GPKG:   {gpkg_path.name}")
                    msg(f"  Layer:  {layer_name}")
                    msg(str(e))
                    msg(traceback.format_exc())

                    overall_summary.append({
                        "block_folder": block_folder_name,
                        "gpkg": gpkg_path.name,
                        "block_layer": layer_name,
                        "k": get_k_suffix(layer_name),
                        "selected_grid_cells": None,
                        "building_intersections": None,
                        "grid_building_block_intersections": None,
                        "block_features_updated": None,
                        "zero_population_features": None,
                        "total_grid_population_seen": None,
                        "total_population_assigned_to_blocks": None,
                        "status": f"ERROR: {str(e)}",
                    })

                    # Keep original layer if this layer failed.
                    original_gdf = read_vector(gpkg_path, layer=layer_name)
                    updated_layers[layer_name] = original_gdf.reset_index(drop=True)

            write_updated_gpkg(gpkg_path, updated_layers)

            msg("Updated GeoPackage written:")
            msg(f"  {gpkg_path}", also_console=True)

        except Exception:
            msg("")
            msg("FAILED on this block folder:")
            msg(traceback.format_exc())

            overall_summary.append({
                "block_folder": block_folder_name,
                "gpkg": gpkg_path.name if gpkg_path else None,
                "block_layer": None,
                "k": None,
                "selected_grid_cells": None,
                "building_intersections": None,
                "grid_building_block_intersections": None,
                "block_features_updated": None,
                "zero_population_features": None,
                "total_grid_population_seen": None,
                "total_population_assigned_to_blocks": None,
                "status": "ERROR at folder level",
            })

    if summary_csv.exists():
        summary_csv.unlink()

    msg("")
    msg("Writing overall summary CSV:")
    msg(f"  {summary_csv}")

    fieldnames = [
        "block_folder",
        "gpkg",
        "block_layer",
        "k",
        "selected_grid_cells",
        "building_intersections",
        "grid_building_block_intersections",
        "block_features_updated",
        "zero_population_features",
        "total_grid_population_seen",
        "total_population_assigned_to_blocks",
        "status",
    ]

    with open(summary_csv, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()

        for row in overall_summary:
            writer.writerow(row)

    msg("")
    msg("All done.", also_console=True)
    msg(f"Layers processed: {len(overall_summary)}", also_console=True)
    msg("Summary written to:", also_console=True)
    msg(f"  {summary_csv}", also_console=True)


if __name__ == "__main__":
    main()


In [ ]:
from pathlib import Path
import re

import pandas as pd
import pyogrio


# ---------------------------------------------------------------------
# User settings
# ---------------------------------------------------------------------

ORIGINAL_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_selection"
)

EXPLODED_ROOT = Path(
    r"E:\_johannesburg\_analysis\heterogeneous_largePop_blocks_MPexplode"
)

GPKG_NAME = "new_blocks_populated.gpkg"

ID_FIELD = "block_id"
POP_FIELD = "population"

# Use None to check all folders in EXPLODED_ROOT.
# Or use a list like ["_401", "_457"] to test selected folders.
CHECK_BLOCK_FOLDERS = None
# CHECK_BLOCK_FOLDERS = ["_401"]

# Differences smaller than this are treated as effectively zero.
TOLERANCE = 1e-6

# If True, write the QA CSVs first and then raise an error whenever any layer
# changes total population beyond TOLERANCE.
FAIL_ON_POPULATION_DIFFERENCE = True

OUT_SUMMARY_CSV = EXPLODED_ROOT / "population_compare_original_vs_MPexplode_summary.csv"
OUT_DETAIL_CSV = EXPLODED_ROOT / "population_compare_original_vs_MPexplode_detail.csv"


# ---------------------------------------------------------------------
# Helper functions
# ---------------------------------------------------------------------

def natural_block_key(path_or_name):
    name = path_or_name.name if hasattr(path_or_name, "name") else str(path_or_name)
    try:
        return 0, int(name.lstrip("_"))
    except ValueError:
        return 1, name.lower()


def list_spatial_layers(gpkg_path):
    layers = pyogrio.list_layers(gpkg_path)
    return [
        str(layer_name)
        for layer_name, geom_type in layers
        if geom_type is not None
    ]


def normalize_layer_name(name):
    """
    ArcGIS may display layers like main.blk_1_401_5,
    while pyogrio may see blk_1_401_5.
    """
    name = str(name)
    if "." in name:
        name = name.split(".")[-1]
    return name.lower()


def pair_layers(original_layers, exploded_layers):
    """
    Pair layers by normalized name. If both GeoPackages only have one spatial
    layer, pair those even if the names do not match exactly.
    """
    pairs = []

    original_norm = {
        normalize_layer_name(name): name
        for name in original_layers
    }

    exploded_norm = {
        normalize_layer_name(name): name
        for name in exploded_layers
    }

    common = sorted(set(original_norm).intersection(exploded_norm))

    for key in common:
        pairs.append((original_norm[key], exploded_norm[key]))

    if not pairs and len(original_layers) == 1 and len(exploded_layers) == 1:
        pairs.append((original_layers[0], exploded_layers[0]))

    return pairs


def read_id_pop(gpkg_path, layer_name):
    df = pyogrio.read_dataframe(
        gpkg_path,
        layer=layer_name,
        columns=[ID_FIELD, POP_FIELD],
        read_geometry=False,
    )

    missing = [c for c in [ID_FIELD, POP_FIELD] if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"Missing required field(s) {missing} in "
            f"{gpkg_path} | {layer_name}"
        )

    df = df[[ID_FIELD, POP_FIELD]].copy()
    df[ID_FIELD] = df[ID_FIELD].astype(str)
    df[POP_FIELD] = pd.to_numeric(df[POP_FIELD], errors="coerce").fillna(0.0)

    return df


def find_parent_id(exploded_id, original_ids):
    """
    If exploded_id is already in the original layer, keep it.
    Otherwise strip right-most underscore suffixes until a match is found.

    Example:
      blk_401_2_0_2_1 -> blk_401_2_0_2
    """
    if exploded_id in original_ids:
        return exploded_id

    candidate = str(exploded_id)

    while "_" in candidate:
        candidate = candidate.rsplit("_", 1)[0]

        if candidate in original_ids:
            return candidate

    return str(exploded_id)


def compare_layer(block_folder, original_gpkg, exploded_gpkg, original_layer, exploded_layer):
    original_df = read_id_pop(original_gpkg, original_layer)
    exploded_df = read_id_pop(exploded_gpkg, exploded_layer)

    original_ids = set(original_df[ID_FIELD].dropna().astype(str))

    exploded_df["parent_block_id"] = exploded_df[ID_FIELD].apply(
        lambda x: find_parent_id(x, original_ids)
    )

    original_agg = (
        original_df
        .groupby(ID_FIELD, dropna=False)[POP_FIELD]
        .sum()
        .reset_index()
        .rename(columns={
            ID_FIELD: "parent_block_id",
            POP_FIELD: "original_population",
        })
    )

    exploded_agg = (
        exploded_df
        .groupby("parent_block_id", dropna=False)[POP_FIELD]
        .sum()
        .reset_index()
        .rename(columns={
            POP_FIELD: "exploded_population",
        })
    )

    compare = original_agg.merge(
        exploded_agg,
        on="parent_block_id",
        how="outer",
    )

    compare["original_population"] = compare["original_population"].fillna(0.0)
    compare["exploded_population"] = compare["exploded_population"].fillna(0.0)

    compare["population_difference"] = (
        compare["exploded_population"] - compare["original_population"]
    )

    compare["absolute_difference"] = compare["population_difference"].abs()

    compare["different"] = compare["absolute_difference"] > TOLERANCE

    compare.insert(0, "block_folder", block_folder)
    compare.insert(1, "original_gpkg", original_gpkg.name)
    compare.insert(2, "exploded_gpkg", exploded_gpkg.name)
    compare.insert(3, "original_layer", original_layer)
    compare.insert(4, "exploded_layer", exploded_layer)

    original_total = float(original_df[POP_FIELD].sum())
    exploded_total = float(exploded_df[POP_FIELD].sum())
    total_difference = exploded_total - original_total

    summary = {
        "block_folder": block_folder,
        "original_gpkg": original_gpkg.name,
        "exploded_gpkg": exploded_gpkg.name,
        "original_layer": original_layer,
        "exploded_layer": exploded_layer,
        "original_feature_count": len(original_df),
        "exploded_feature_count": len(exploded_df),
        "original_total_population": original_total,
        "exploded_total_population": exploded_total,
        "total_population_difference": total_difference,
        "absolute_total_difference": abs(total_difference),
        "same_total_population": abs(total_difference) <= TOLERANCE,
        "parent_block_ids_checked": len(compare),
        "parent_block_ids_with_difference": int(compare["different"].sum()),
        "max_parent_absolute_difference": float(compare["absolute_difference"].max())
            if len(compare) else 0.0,
    }

    return summary, compare


# ---------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------

if CHECK_BLOCK_FOLDERS is None:
    block_folders = sorted(
        [p.name for p in EXPLODED_ROOT.iterdir() if p.is_dir()],
        key=natural_block_key,
    )
else:
    block_folders = sorted(CHECK_BLOCK_FOLDERS, key=natural_block_key)

summary_rows = []
detail_frames = []

print(f"Checking {len(block_folders)} block folder(s)...\n")

for block_folder in block_folders:

    original_folder = ORIGINAL_ROOT / block_folder
    exploded_folder = EXPLODED_ROOT / block_folder

    original_gpkg = original_folder / GPKG_NAME
    exploded_gpkg = exploded_folder / GPKG_NAME

    if not original_gpkg.exists():
        print(f"[MISSING ORIGINAL] {block_folder} | {original_gpkg}")
        continue

    if not exploded_gpkg.exists():
        print(f"[MISSING EXPLODED] {block_folder} | {exploded_gpkg}")
        continue

    try:
        original_layers = list_spatial_layers(original_gpkg)
        exploded_layers = list_spatial_layers(exploded_gpkg)
        layer_pairs = pair_layers(original_layers, exploded_layers)

        if not layer_pairs:
            print(f"[NO MATCHING LAYERS] {block_folder}")
            print(f"  Original layers: {original_layers}")
            print(f"  Exploded layers: {exploded_layers}")
            continue

        for original_layer, exploded_layer in layer_pairs:
            summary, detail = compare_layer(
                block_folder=block_folder,
                original_gpkg=original_gpkg,
                exploded_gpkg=exploded_gpkg,
                original_layer=original_layer,
                exploded_layer=exploded_layer,
            )

            summary_rows.append(summary)
            detail_frames.append(detail)

            if summary["same_total_population"]:
                print(
                    f"[SAME]      {block_folder} | "
                    f"{original_layer} -> {exploded_layer} | "
                    f"total diff = {summary['total_population_difference']:.12f}"
                )
            else:
                print(
                    f"[DIFFERENT] {block_folder} | "
                    f"{original_layer} -> {exploded_layer} | "
                    f"original = {summary['original_total_population']:.6f}, "
                    f"exploded = {summary['exploded_total_population']:.6f}, "
                    f"diff = {summary['total_population_difference']:.6f}"
                )

                changed_detail = detail[detail["different"]].copy()

                if not changed_detail.empty:
                    print("  Parent block_id differences:")
                    for _, r in changed_detail.sort_values(
                        "absolute_difference",
                        ascending=False,
                    ).head(10).iterrows():
                        print(
                            f"    {r['parent_block_id']}: "
                            f"original={r['original_population']:.6f}, "
                            f"exploded={r['exploded_population']:.6f}, "
                            f"diff={r['population_difference']:.6f}"
                        )

    except Exception as exc:
        print(f"[ERROR] {block_folder}")
        print(f"  {exc}")


summary_df = pd.DataFrame(summary_rows)

if detail_frames:
    detail_df = pd.concat(detail_frames, ignore_index=True)
else:
    detail_df = pd.DataFrame()

summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
detail_df.to_csv(OUT_DETAIL_CSV, index=False)

print("\n" + "=" * 78)
print("DONE")
print("=" * 78)
print(f"Summary CSV: {OUT_SUMMARY_CSV}")
print(f"Detail CSV:  {OUT_DETAIL_CSV}")

if not summary_df.empty:
    n_same = int(summary_df["same_total_population"].sum())
    n_total = len(summary_df)

    print()
    print(f"Layer comparisons with same total population: {n_same} / {n_total}")

    different = summary_df[~summary_df["same_total_population"]].copy()

    if not different.empty:
        print("\nLayers with population differences:")
        for _, r in different.sort_values(
            "absolute_total_difference",
            ascending=False,
        ).iterrows():
            print(
                f"  {r['block_folder']} | "
                f"{r['original_layer']} -> {r['exploded_layer']} | "
                f"diff = {r['total_population_difference']:.6f}"
            )

    if FAIL_ON_POPULATION_DIFFERENCE and not different.empty:
        raise RuntimeError(
            f"{len(different):,} layer comparison(s) changed total population "
            f"beyond tolerance {TOLERANCE}. Review {OUT_SUMMARY_CSV} and "
            f"{OUT_DETAIL_CSV} before continuing."
        )